In [ ]:
import polars as pl
import numpy as np
from datasets import load_dataset

In [ ]:
ds = load_dataset("Jsevisal/go_emotions_wheel")

In [ ]:
df_train = pl.from_dicts(ds['train'][:])
df_test = pl.from_dicts(ds['test'][:])
df_validation = pl.from_dicts(ds['validation'][:])

In [ ]:
df_train = df_train.filter(
    ~(
        pl.col("labels").list.contains(1) |
        pl.col("labels").list.contains(5) |
        pl.col("labels").list.contains(6)
    )
)

In [ ]:
df_test = df_test.filter(
    ~(
        pl.col("labels").list.contains(1) |
        pl.col("labels").list.contains(5) |
        pl.col("labels").list.contains(6)
    )
)

In [ ]:
df_validation = df_validation.filter(
    ~(
        pl.col("labels").list.contains(1) |
        pl.col("labels").list.contains(5) |
        pl.col("labels").list.contains(6)
    )
)

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model_st = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
X_train = model_st.encode(df_train['text'].to_list(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_test  = model_st.encode(df_test['text'].to_list(),  batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_val = model_st.encode(df_validation['text'].to_list(),  batch_size=64, show_progress_bar=True, normalize_embeddings=True)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier

In [ ]:
y_train_raw = df_train['labels'].to_list()
y_test_raw = df_test['labels'].to_list()
y_val_raw = df_validation['labels'].to_list()

all_labels_combined = y_train_raw + y_test_raw + y_val_raw
all_unique_label_ids = sorted(list(set(item for sublist in all_labels_combined for item in sublist)))

In [ ]:
mlb = MultiLabelBinarizer(classes=all_unique_label_ids)
mlb.fit(all_labels_combined)

In [ ]:
y_train = mlb.transform(y_train_raw)
y_test = mlb.transform(y_test_raw)
y_val = mlb.transform(y_val_raw)

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
model = OneVsRestClassifier(
    LogisticRegression(C=1.0, class_weight='balanced', solver='lbfgs', max_iter=500)
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import hamming_loss, jaccard_score

print("Hamming Loss:", hamming_loss(y_test, y_pred))
print("Jaccard (macro):", jaccard_score(y_test, y_pred, average='macro'))

In [ ]:
label_dict = {
  0: 'joy',
  2: 'fear',
  3: 'surprise',
  4: 'sadness',
  7: 'anger',
  8: 'disgust'
}

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=[label_dict[c] for c in mlb.classes_], zero_division=0))

In [ ]:
from sklearn.metrics import accuracy_score
print("Subset Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
import pandas as pd
label_counts = pd.DataFrame(y_train.sum(axis=0), index=[label_dict[c] for c in mlb.classes_], columns=['count'])
print(label_counts.sort_values('count', ascending=False))

In [ ]:
y_pred_proba = model.predict_proba(X_test)   # list of arrays, one per label
thresholds = [0.3, 0.25, 0.4, 0.35, 0.2, 0.15]   # example – order by label_dict

# Initialize y_pred_bin with the same shape as y_pred_proba (num_samples, num_labels)
y_pred_bin = np.zeros_like(y_pred_proba)
for i in range(len(thresholds)):
    # Apply the threshold to the i-th column (label) of y_pred_proba
    y_pred_bin[:, i] = (y_pred_proba[:, i] >= thresholds[i]).astype(int)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'estimator__C': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    'estimator__penalty': ['l2'],           # l1 can work but needs liblinear solver
    'estimator__class_weight': ['balanced', None],
    'estimator__max_iter': [1000],
}

grid = GridSearchCV(
    model, param_grid, cv=3, scoring='f1_macro', n_jobs=-1
)
grid.fit(X_train, y_train)
print(grid.best_params_)

In [ ]:
import optuna

def objective(trial):
    C = trial.suggest_float('C', 0.01, 10.0, log=True)
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga'])

    model = OneVsRestClassifier(LogisticRegression(C=C, class_weight=class_weight, solver=solver, max_iter=1000))
    model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)
    score = jaccard_score(y_val, y_pred_val, average='macro')  # O f1_score(average='macro')
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)  # Ajusta trials
print(study.best_params)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def find_best_thresholds(y_true, y_proba, num_labels):
    best_thresholds = []
    for i in range(num_labels):
        thresholds = np.arange(0.1, 0.9, 0.05)
        best_score = 0
        best_th = 0.5
        for th in thresholds:
            y_pred_i = (y_proba[:, i] >= th).astype(int)
            score = f1_score(y_true[:, i], y_pred_i)
            if score > best_score:
                best_score = score
                best_th = th
        best_thresholds.append(best_th)
    return best_thresholds

# Después de fit
y_pred_proba_val = model.predict_proba(X_val)
thresholds = find_best_thresholds(y_val, y_pred_proba_val, y_val.shape[1])
print(thresholds)  # Usa estos en test

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import hamming_loss, jaccard_score

configs = [
    {"name": "Base", "C": 1.0, "class_weight": "balanced"},
    {"name": "GridSearch", "C": 10.0, "class_weight": None},
    {"name": "Optuna", "C": 0.0100936, "class_weight": "balanced"}
]

results = []

for config in configs:
    print(f"Entrenando modelo: {config['name']}...")
    clf = OneVsRestClassifier(
        LogisticRegression(C=config['C'], class_weight=config['class_weight'], solver='lbfgs', max_iter=1000)
    )
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    
    h_loss = hamming_loss(y_test, y_pred)
    j_score = jaccard_score(y_test, y_pred, average='macro')
    
    results.append({
        "Modelo": config['name'],
        "Hamming Loss": h_loss,
        "Jaccard Macro": j_score
    })


import pandas as pd
df_results = pd.DataFrame(results)
print("\nComparativa Final de Modelos (Embeddings BGE):")
print(df_results)

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model_gridsearch = OneVsRestClassifier(
    LogisticRegression(
        C=10.0, 
        class_weight=None, 
        solver='lbfgs', 
        max_iter=1000
    )
)

# 2. Entrenar con los datos de entrenamiento (BGE embeddings)
model_gridsearch.fit(X_train, y_train)

# 3. Realizar predicciones en el set de prueba
y_pred_gs = model_gridsearch.predict(X_test)

# 4. Generar el reporte de clasificación
print("Reporte de Clasificación - Hiperparámetros GridSearch:")
print(classification_report(
    y_test, 
    y_pred_gs, 
    target_names=[label_dict[c] for c in mlb.classes_], 
    zero_division=0
))

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Definimos un pipeline más robusto
# Solo usamos SMOTE para evitar el conflicto de ratios del UnderSampler
resampling_pipeline = ImbPipeline([
    ('smote', SMOTE(sampling_strategy='auto', random_state=42)),
    ('classifier', LogisticRegression(C=10.0, max_iter=1000, solver='lbfgs'))
])

# 2. Envolver en OneVsRest
model_smote = OneVsRestClassifier(resampling_pipeline)

# 3. Entrenar
print("Entrenando modelo con SMOTE optimizado...")
model_smote.fit(X_train, y_train)

# 4. Predicción y Reporte
y_pred_smote = model_smote.predict(X_test)

print("\nReporte de Clasificación (SMOTE):")
print(classification_report(
    y_test, 
    y_pred_smote, 
    target_names=[label_dict[c] for c in mlb.classes_], 
    zero_division=0
))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

def find_best_thresholds(y_true, y_proba, num_labels):
    """Encuentra el umbral óptimo por cada etiqueta en el set de validación."""
    best_thresholds = []
    for i in range(num_labels):
        thresholds = np.arange(0.05, 0.85, 0.05) # Probamos desde 0.05 hasta 0.80
        best_score = 0
        best_th = 0.5
        for th in thresholds:
            y_pred_i = (y_proba[:, i] >= th).astype(int)
            score = f1_score(y_true[:, i], y_pred_i)
            if score > best_score:
                best_score = score
                best_th = th
        best_thresholds.append(best_th)
    return best_thresholds

# 1. Obtener probabilidades en VALIDACIÓN para calcular los umbrales
y_proba_val = model_smote.predict_proba(X_val)
opt_thresholds = find_best_thresholds(y_val, y_proba_val, y_val.shape[1])

print("Umbrales óptimos encontrados (por etiqueta):")
for i, label in enumerate([label_dict[c] for c in mlb.classes_]):
    print(f"{label}: {opt_thresholds[i]:.2f}")

# 2. Obtener probabilidades en TEST para la evaluación final
y_proba_test = model_smote.predict_proba(X_test)

# 3. Aplicar los umbrales personalizados a las probabilidades de test
y_pred_custom = np.zeros_like(y_proba_test)
for i in range(len(opt_thresholds)):
    y_pred_custom[:, i] = (y_proba_test[:, i] >= opt_thresholds[i]).astype(int)

# 4. Reporte final
print("\nReporte de Clasificación Final (SMOTE + Thresholds Optimizados):")
print(classification_report(
    y_test, 
    y_pred_custom, 
    target_names=[label_dict[c] for c in mlb.classes_], 
    zero_division=0
))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

y_proba_test = model.predict_proba(X_test) 

# Diccionarios para guardar los Falsos Positivos, Verdaderos Positivos y el AUC
fpr = dict()
tpr = dict()
roc_auc = dict()
n_classes = y_test.shape[1]

# 2. Calcular ROC y AUC por cada clase
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_proba_test[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 3. Configurar la gráfica
plt.figure(figsize=(10, 8))
colores = ['blue', 'red', 'green', 'purple', 'orange', 'brown', 'pink']

# 4. Graficar cada curva
for i, color in zip(range(n_classes), colores):
    nombre_clase = label_dict[mlb.classes_[i]]
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'ROC {nombre_clase} (AUC = {roc_auc[i]:0.2f})')

# Línea diagonal de referencia (clasificador aleatorio)
plt.plot([0, 1], [0, 1], 'k--', lw=2)

# Ajustes visuales
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (False Positive Rate)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (True Positive Rate)', fontsize=12)
plt.title('Curva ROC por Emoción', fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)

# Mostrar la gráfica
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, hamming_loss, jaccard_score, classification_report

# 1. Definir función para encontrar el mejor umbral por emoción optimizando F1
def find_best_thresholds(y_true, y_proba, num_labels):
    best_thresholds = []
    for i in range(num_labels):
        # Probamos umbrales desde 0.10 hasta 0.90
        thresholds = np.arange(0.1, 0.9, 0.05)
        best_score = 0
        best_th = 0.5
        for th in thresholds:
            # Predicción binaria con el umbral actual
            y_pred_i = (y_proba[:, i] >= th).astype(int)
            # Calculamos F1-Score para esta etiqueta
            score = f1_score(y_true[:, i], y_pred_i)
            if score > best_score:
                best_score = score
                best_th = th
        best_thresholds.append(best_th)
    return best_thresholds

# 2. Predecir probabilidades en VALIDACIÓN
y_proba_val = model.predict_proba(X_val)

# 3. Calcular los umbrales óptimos
opt_thresholds = find_best_thresholds(y_val, y_proba_val, y_val.shape[1])

print("--- UMBRALES ÓPTIMOS ENCONTRADOS ---")
for i, label in enumerate([label_dict[c] for c in mlb.classes_]):
    print(f"{label}: {opt_thresholds[i]:.2f}")
print("-" * 36)

# 4. Predecir probabilidades en el set de PRUEBA (TEST)
y_proba_test = model.predict_proba(X_test)

# 5. Aplicar los umbrales personalizados a las predicciones de TEST
y_pred_tuned = np.zeros_like(y_proba_test)
for i in range(len(opt_thresholds)):
    y_pred_tuned[:, i] = (y_proba_test[:, i] >= opt_thresholds[i]).astype(int)

# 6. Calcular y mostrar las MÉTRICAS FINALES
print("\n--- MÉTRICAS GLOBALES (CON UMBRALES AJUSTADOS) ---")
print(f"Hamming Loss:    {hamming_loss(y_test, y_pred_tuned):.4f} (↓ menor es mejor)")
print(f"Jaccard (macro): {jaccard_score(y_test, y_pred_tuned, average='macro'):.4f} (↑ mayor es mejor)")

print("\n--- REPORTE DE CLASIFICACIÓN FINAL ---")
print(classification_report(
    y_test, 
    y_pred_tuned, 
    target_names=[label_dict[c] for c in mlb.classes_], 
    zero_division=0
))

In [ ]:
import numpy as np
from sklearn.metrics import fbeta_score, hamming_loss, jaccard_score, classification_report

def find_balanced_thresholds(y_true, y_proba, num_labels, beta=0.5):
    best_thresholds = []
    for i in range(num_labels):
        # Empezamos a buscar desde 0.25 para evitar umbrales excesivamente bajos
        thresholds = np.arange(0.25, 0.85, 0.05)
        best_score = 0
        best_th = 0.5
        for th in thresholds:
            y_pred_i = (y_proba[:, i] >= th).astype(int)
            # fbeta_score con beta=0.5 penaliza fuertemente los falsos positivos
            score = fbeta_score(y_true[:, i], y_pred_i, beta=beta)
            if score > best_score:
                best_score = score
                best_th = th
        best_thresholds.append(best_th)
    return best_thresholds

# 1. Buscar los nuevos umbrales estrictos en VALIDACIÓN
y_proba_val = model.predict_proba(X_val)
opt_thresholds_strict = find_balanced_thresholds(y_val, y_proba_val, y_val.shape[1], beta=0.5)

print("--- NUEVOS UMBRALES (Penalizando Falsos Positivos) ---")
for i, label in enumerate([label_dict[c] for c in mlb.classes_]):
    print(f"{label}: {opt_thresholds_strict[i]:.2f}")
print("-" * 54)

# 2. Predecir en TEST con los nuevos umbrales
y_proba_test = model.predict_proba(X_test)
y_pred_balanced = np.zeros_like(y_proba_test)

for i in range(len(opt_thresholds_strict)):
    y_pred_balanced[:, i] = (y_proba_test[:, i] >= opt_thresholds_strict[i]).astype(int)

# 3. Mostrar las Métricas Finales
print("\n--- MÉTRICAS FINALES BALANCEADAS ---")
print(f"Hamming Loss:    {hamming_loss(y_test, y_pred_balanced):.4f}")
print(f"Jaccard (macro): {jaccard_score(y_test, y_pred_balanced, average='macro'):.4f}")

print("\n--- REPORTE DE CLASIFICACIÓN FINAL ---")
print(classification_report(
    y_test, 
    y_pred_balanced, 
    target_names=[label_dict[c] for c in mlb.classes_], 
    zero_division=0
))

In [ ]:
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import fbeta_score, roc_curve, auc

# ==========================================
# 2. BUSCAR NUEVOS UMBRALES (F-Beta 0.5)
# ==========================================
def find_balanced_thresholds(y_true, y_proba, num_labels, beta=0.5):
    best_thresholds = []
    for i in range(num_labels):
        thresholds = np.arange(0.25, 0.85, 0.05)
        best_score = 0
        best_th = 0.5
        for th in thresholds:
            y_pred_i = (y_proba[:, i] >= th).astype(int)
            score = fbeta_score(y_true[:, i], y_pred_i, beta=beta)
            if score > best_score:
                best_score = score
                best_th = th
        best_thresholds.append(best_th)
    return best_thresholds

# Predecir probabilidades calibradas en validación y test
y_proba_val_cal = calibrated_model.predict_proba(X_val)
y_proba_test_cal = calibrated_model.predict_proba(X_test)

# Encontrar umbrales óptimos penalizando falsos positivos
opt_thresholds = find_balanced_thresholds(y_val, y_proba_val_cal, y_val.shape[1], beta=0.5)

print("--- UMBRALES ÓPTIMOS (PROBABILIDADES CALIBRADAS) ---")
for i, label in enumerate([label_dict[c] for c in mlb.classes_]):
    print(f"{label}: {opt_thresholds[i]:.2f}")
print("-" * 52)

# ==========================================
# 3. GRAFICAR CURVA ROC Y MARCAR UMBRALES
# ==========================================
plt.figure(figsize=(12, 9))
colores = ['blue', 'red', 'green', 'purple', 'orange', 'brown', 'pink']
n_classes = y_test.shape[1]

for i, color in zip(range(n_classes), colores):
    nombre_clase = label_dict[mlb.classes_[i]]
    
    # Calcular Curva ROC y AUC para la clase i
    fpr_i, tpr_i, roc_thresholds = roc_curve(y_test[:, i], y_proba_test_cal[:, i])
    roc_auc_i = auc(fpr_i, tpr_i)
    
    # Trazar la línea de la curva
    plt.plot(fpr_i, tpr_i, color=color, lw=2, 
             label=f'ROC {nombre_clase} (AUC = {roc_auc_i:0.2f})')
    
    # Encontrar el punto exacto en la curva que corresponde a tu umbral penalizado
    # Buscamos el índice del umbral de la curva ROC más cercano a nuestro umbral óptimo
    idx_umbral = np.argmin(np.abs(roc_thresholds - opt_thresholds[i]))
    
    # Dibujar un punto grande donde cae el umbral elegido
    plt.plot(fpr_i[idx_umbral], tpr_i[idx_umbral], marker='o', color=color, 
             markersize=10, markeredgecolor='black', 
             label=f'Umbral {nombre_clase} ({opt_thresholds[i]:.2f})')

# Línea diagonal (aleatoria)
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Clasificador Aleatorio')

# Formato final de la gráfica
plt.xlim([-0.02, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR) -> Penalti de Falsas Alarmas', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (TPR) -> Recall', fontsize=12)
plt.title('Curva ROC de Probabilidades Calibradas con Umbrales (F0.5)', fontsize=15, fontweight='bold')
plt.legend(loc="lower right", fontsize=10, ncol=2)
plt.grid(alpha=0.3)
nombre_imagen = 'roc_curve_calibrated.png'
plt.savefig(nombre_imagen, dpi=300, bbox_inches='tight')
print(f" Imagen de la curva ROC guardada como: '{nombre_imagen}'")

plt.show()